# Vaccination Coverage Analysis - Data Cleaning

## Project Overview
This notebook implements the **Question → Data → Insight** data science lifecycle to analyze vaccination coverage data from India's health survey.

**Core Question:** Which regions are under-vaccinated, where are rollout inconsistencies occurring, and how are vaccination trends changing over time?

## Data Science Lifecycle Implementation

### 1. Question (Already Defined)
- **Primary Goal**: Identify under-vaccinated regions
- **Secondary Goals**: 
  - Detect rollout inconsistencies across districts
  - Understand vaccination coverage patterns
  - Analyze public vs private healthcare delivery

### 2. Data (This Notebook)
- Load and examine vaccination coverage data
- Clean and validate data quality
- Prepare metrics for analysis
- Handle missing values and outliers

### 3. Insight (Next Notebooks)
- Identify regional disparities in vaccination coverage
- Detect inconsistencies in rollout patterns
- Provide actionable recommendations for public health policy

In [1]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)

print("Libraries imported successfully!")
print(f"Pandas version: {pd.__version__}")
print(f"NumPy version: {np.__version__}")

Libraries imported successfully!
Pandas version: 3.0.1
NumPy version: 2.4.2


In [2]:
# Define file paths
data_path = Path("../data/raw/datafile.csv")
processed_path = Path("../data/processed/")
processed_path.mkdir(exist_ok=True)

# Load the vaccination coverage data
print("Loading vaccination coverage data...")
df = pd.read_csv(data_path)

# Display basic information about the dataset
print(f"\nDataset shape: {df.shape}")
print(f"Columns: {len(df.columns)}")
print(f"Districts: {len(df)}")
print("\n" + "="*50)
print("FIRST FEW COLUMNS:")
print("="*50)
for i, col in enumerate(df.columns[:5]):
    print(f"{i+1}. {col}")

Loading vaccination coverage data...

Dataset shape: (706, 109)
Columns: 109
Districts: 706

FIRST FEW COLUMNS:
1. District Names
2. State/UT
3. Number of Households surveyed
4. Number of Women age 15-49 years interviewed
5. Number of Men age 15-54 years interviewed


In [3]:
# Identify vaccination-related columns
vaccination_keywords = ['vaccin', 'BCG', 'polio', 'measles', 'DPT', 'penta', 'rotavirus', 'hepatitis']

vaccination_cols = []
for col in df.columns:
    col_lower = col.lower()
    if any(keyword in col_lower for keyword in vaccination_keywords):
        vaccination_cols.append(col)

print(f"Found {len(vaccination_cols)} vaccination-related columns:")
print("="*60)
for i, col in enumerate(vaccination_cols, 1):
    print(f"{i:2d}. {col}")
    
print("\n" + "="*60)
print("GEOGRAPHIC IDENTIFIERS:")
print("="*60)
geographic_cols = ['District Names', 'State/UT']
for col in geographic_cols:
    if col in df.columns:
        print(f"✓ {col}")
        unique_count = df[col].nunique()
        print(f"  → {unique_count} unique values")
    else:
        print(f"✗ {col} not found")

Found 10 vaccination-related columns:
 1. Children age 12-23 months fully vaccinated based on information from either vaccination card or mother's recall11 (%)
 2. Children age 12-23 months fully vaccinated based on information from vaccination card only12 (%)
 3. Children age 12-23 months who have received 3 doses of polio vaccine13 (%)
 4. Children age 12-23 months who have received 3 doses of penta or DPT vaccine (%)
 5. Children age 12-23 months who have received the first dose of measles-containing vaccine (MCV) (%)
 6. Children age 24-35 months who have received a second dose of measles-containing vaccine (MCV) (%)
 7. Children age 12-23 months who have received 3 doses of rotavirus vaccine14 (%)
 8. Children age 12-23 months who have received 3 doses of penta or hepatitis B vaccine (%)
 9. Children age 12-23 months who received most of their vaccinations in a public health facility (%)
10. Children age 12-23 months who received most of their vaccinations in a private health faci

In [4]:
# Create simplified column mappings for easier analysis
vaccination_mapping = {
    "Children age 12-23 months fully vaccinated based on information from either vaccination card or mother's recall11 (%)": "full_vaccination_any_source",
    "Children age 12-23 months fully vaccinated based on information from vaccination card only12 (%)": "full_vaccination_card_only",
    "Children age 12-23 months who have received BCG (%)": "bcg_vaccination",
    "Children age 12-23 months who have received 3 doses of polio vaccine13 (%)": "polio_3_doses",
    "Children age 12-23 months who have received 3 doses of penta or DPT vaccine (%)": "dpt_3_doses",
    "Children age 12-23 months who have received the first dose of measles-containing vaccine (MCV) (%)": "measles_first_dose",
    "Children age 24-35 months who have received a second dose of measles-containing vaccine (MCV) (%)": "measles_second_dose",
    "Children age 12-23 months who have received 3 doses of rotavirus vaccine14 (%)": "rotavirus_3_doses",
    "Children age 12-23 months who have received 3 doses of penta or hepatitis B vaccine (%)": "hepatitis_3_doses",
    "Children age 9-35 months who received a vitamin A dose in the last 6 months (%)": "vitamin_a_supplementation",
    "Children age 12-23 months who received most of their vaccinations in a public health facility (%)": "vaccinations_public_facility",
    "Children age 12-23 months who received most of their vaccinations in a private health facility (%)": "vaccinations_private_facility"
}

# Create a subset with key columns for analysis
key_columns = ['District Names', 'State/UT'] + list(vaccination_mapping.keys())

# Check which columns exist in our dataset
available_columns = [col for col in key_columns if col in df.columns]
missing_columns = [col for col in key_columns if col not in df.columns]

print(f"Available key columns: {len(available_columns)}")
print(f"Missing columns: {len(missing_columns)}")

if missing_columns:
    print(f"\nMissing columns:")
    for col in missing_columns:
        print(f"  ✗ {col}")

# Create working dataset
df_vaccination = df[available_columns].copy()

print(f"\nWorking dataset shape: {df_vaccination.shape}")

Available key columns: 14
Missing columns: 0

Working dataset shape: (706, 14)


In [5]:
# Rename columns for easier analysis
rename_dict = {'District Names': 'district', 'State/UT': 'state'}

# Add vaccination column renames for those that exist
for original_col, short_name in vaccination_mapping.items():
    if original_col in df_vaccination.columns:
        rename_dict[original_col] = short_name

df_vaccination = df_vaccination.rename(columns=rename_dict)

print("Column renaming completed!")
print(f"\nNew column names:")
for i, col in enumerate(df_vaccination.columns, 1):
    print(f"{i:2d}. {col}")
    
# Display first few rows
print(f"\nFirst 3 rows:")
print("="*80)
print(df_vaccination.head(3))

Column renaming completed!

New column names:
 1. district
 2. state
 3. full_vaccination_any_source
 4. full_vaccination_card_only
 5. bcg_vaccination
 6. polio_3_doses
 7. dpt_3_doses
 8. measles_first_dose
 9. measles_second_dose
10. rotavirus_3_doses
11. hepatitis_3_doses
12. vitamin_a_supplementation
13. vaccinations_public_facility
14. vaccinations_private_facility

First 3 rows:
                  district                      state full_vaccination_any_source full_vaccination_card_only bcg_vaccination polio_3_doses dpt_3_doses measles_first_dose measles_second_dose rotavirus_3_doses hepatitis_3_doses vitamin_a_supplementation vaccinations_public_facility vaccinations_private_facility
0                 Nicobars  Andaman & Nicobar Islands                      (64.2)                     (94.1)          (80.4)        (69.1)      (71.9)             (67.3)              (20.7)             (3.1)            (68.6)                     94.9                       (100.0)                    

In [6]:
# Data Quality Assessment
print("DATA QUALITY ASSESSMENT")
print("="*60)

# Check for missing values
print("1. Missing Values Analysis:")
missing_summary = df_vaccination.isnull().sum()
total_rows = len(df_vaccination)

for col in df_vaccination.columns:
    missing_count = missing_summary[col]
    missing_pct = (missing_count / total_rows) * 100
    status = "✓" if missing_count == 0 else "⚠" if missing_pct < 10 else "✗"
    print(f"   {status} {col}: {missing_count} ({missing_pct:.1f}%)")

# Check data types
print(f"\n2. Data Types:")
print(df_vaccination.dtypes)

# Check for invalid values (vaccination percentages should be 0-100)
print(f"\n3. Data Range Validation (Vaccination %):")
vaccination_cols = [col for col in df_vaccination.columns if col not in ['district', 'state']]

for col in vaccination_cols:
    if col in df_vaccination.columns:
        # Convert to numeric, replacing non-numeric values with NaN
        df_vaccination[col] = pd.to_numeric(df_vaccination[col], errors='coerce')
        
        valid_range = df_vaccination[col].between(0, 100, inclusive='both')
        invalid_count = (~valid_range).sum()
        
        min_val = df_vaccination[col].min()
        max_val = df_vaccination[col].max()
        
        status = "✓" if invalid_count == 0 else "⚠"
        print(f"   {status} {col}: Range [{min_val:.1f}, {max_val:.1f}], Invalid: {invalid_count}")

print(f"\nDataset summary after quality check:")
print(f"Total districts: {len(df_vaccination)}")
print(f"Total states: {df_vaccination['state'].nunique()}")
print(f"Vaccination metrics: {len(vaccination_cols)}")

DATA QUALITY ASSESSMENT
1. Missing Values Analysis:
   ✓ district: 0 (0.0%)
   ✓ state: 0 (0.0%)
   ✓ full_vaccination_any_source: 0 (0.0%)
   ✓ full_vaccination_card_only: 0 (0.0%)
   ✓ bcg_vaccination: 0 (0.0%)
   ✓ polio_3_doses: 0 (0.0%)
   ✓ dpt_3_doses: 0 (0.0%)
   ✓ measles_first_dose: 0 (0.0%)
   ✓ measles_second_dose: 0 (0.0%)
   ✓ rotavirus_3_doses: 0 (0.0%)
   ✓ hepatitis_3_doses: 0 (0.0%)
   ✓ vitamin_a_supplementation: 0 (0.0%)
   ✓ vaccinations_public_facility: 0 (0.0%)
   ✓ vaccinations_private_facility: 0 (0.0%)

2. Data Types:
district                         str
state                            str
full_vaccination_any_source      str
full_vaccination_card_only       str
bcg_vaccination                  str
polio_3_doses                    str
dpt_3_doses                      str
measles_first_dose               str
measles_second_dose              str
rotavirus_3_doses                str
hepatitis_3_doses                str
vitamin_a_supplementation        str
vaccin

In [7]:
# Save cleaned dataset
output_file = processed_path / "vaccination_coverage_clean.csv"
df_vaccination.to_csv(output_file, index=False)

print("✓ Cleaned dataset saved successfully!")
print(f"File location: {output_file}")
print(f"File size: {output_file.stat().st_size / 1024:.1f} KB")

# Create data summary for next notebooks
summary_stats = {
    'total_districts': len(df_vaccination),
    'total_states': df_vaccination['state'].nunique(),
    'vaccination_metrics': len(vaccination_cols),
    'data_file': str(output_file)
}

# Save summary
import json
summary_file = processed_path / "data_summary.json"
with open(summary_file, 'w') as f:
    json.dump(summary_stats, f, indent=2)

print(f"\n✓ Data summary saved: {summary_file}")
print("\nNext steps:")
print("1. Run EDA notebook (02_eda_visualizations.ipynb)")
print("2. Identify under-vaccinated regions")
print("3. Detect rollout inconsistencies")
print("4. Generate insights for public health policy")

✓ Cleaned dataset saved successfully!
File location: ../data/processed/vaccination_coverage_clean.csv
File size: 45.1 KB

✓ Data summary saved: ../data/processed/data_summary.json

Next steps:
1. Run EDA notebook (02_eda_visualizations.ipynb)
2. Identify under-vaccinated regions
3. Detect rollout inconsistencies
4. Generate insights for public health policy
